# Preliminary BRB-seq expression PCA

This notebook only reads the deep FastReadCounter matrices, maps barcodes to sample IDs, applies DESeq2's variance-stabilizing transform, and makes separate gill and liver PCA plots. No differential-expression model is fitted here.

The compiled sample metadata are used for labels and annotations. `2024_DJR_196` is intentionally retained with unknown sex so its position can be inspected in the gill PCA.

In [ ]:
library(DESeq2)
library(ggplot2)

# Edit these paths only if the repository/output location changes.
repo_root <- "/scratch/ztobias/Gacu_BC_txmics"
count_root <- file.path(repo_root, "outputs", "matrix")
metadata_file <- file.path(repo_root, "metadata", "derived", "sample_tracking.tsv")
barcode_file <- file.path(repo_root, "metadata", "brb_sample_barcodes.tsv")

tracking <- read.delim(metadata_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)
brb_barcodes <- read.delim(barcode_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)

tracking <- tracking[grepl("^(2024|2025)_DJR_", tracking$sample_id), ]
brb_barcodes <- brb_barcodes[grepl("^(2024|2025)_DJR_", brb_barcodes$sample_id), ]
head(tracking[, c("sample_id", "population", "ecotype", "field_sex", "true_sex")])

## Read and map the four deep count matrices

Each year/tissue run has its own matrix. The 2025 gill and liver matrices already represent the independent tissue run plus the combined lane, because those FASTQs were concatenated before STAR.

In [ ]:
read_count_matrix <- function(path) {
  if (!file.exists(path)) stop("Missing count matrix: ", path)
  x <- read.delim(path, header = TRUE, row.names = 1,
    check.names = FALSE, stringsAsFactors = FALSE)
  x <- as.matrix(x)
  storage.mode(x) <- "numeric"
  x
}

matrix_file <- function(tissue, year) {
  file.path(count_root, paste0(tissue, "_", year,
    "_deep_fastreadcounter"), "counts.txt")
}

map_matrix_to_samples <- function(x, tissue, year) {
  map <- brb_barcodes[brb_barcodes$tissue == tissue &
    brb_barcodes$year == as.integer(year), c("sample_id", "barcode")]
  map <- map[!duplicated(map$barcode), ]
  sample_names <- map$sample_id[match(colnames(x), map$barcode)]
  if (anyNA(sample_names)) {
    cat("Unmapped barcodes for ", tissue, " ", year, ":\n")
    print(colnames(x)[is.na(sample_names)])
  }
  keep <- !is.na(sample_names)
  x <- x[, keep, drop = FALSE]
  colnames(x) <- sample_names[keep]
  x
}

read_tissue_counts <- function(tissue) {
  mats <- lapply(c("2024", "2025"), function(year) {
    x <- read_count_matrix(matrix_file(tissue, year))
    map_matrix_to_samples(x, tissue, year)
  })
  genes <- Reduce(intersect, lapply(mats, rownames))
  mats <- lapply(mats, function(x) x[genes, , drop = FALSE])
  out <- do.call(cbind, mats)
  if (anyDuplicated(colnames(out))) stop("Duplicate sample IDs in ", tissue, " matrix")
  out
}

gill_counts <- read_tissue_counts("gill")
liver_counts <- read_tissue_counts("liver")
dim(gill_counts)
dim(liver_counts)

In [ ]:
make_coldata <- function(counts) {
  md <- tracking[match(colnames(counts), tracking$sample_id), , drop = FALSE]
  if (anyNA(md$sample_id)) stop("Count matrix sample missing from compiled metadata")
  rownames(md) <- md$sample_id
  md
}

gill_metadata <- make_coldata(gill_counts)
liver_metadata <- make_coldata(liver_counts)

# Unknown sex is retained as NA. Do not replace it before inspecting PCA.
gill_metadata[gill_metadata$sample_id == "2024_DJR_196",
  c("sample_id", "field_sex", "true_sex", "population", "ecotype")]

table(gill_metadata$year, useNA = "ifany")
table(liver_metadata$year, useNA = "ifany")

## DESeq2 variance stabilization

This uses the same basic logic as the 2024 analysis: remove genes with no useful counts, build a DESeq2 object, and use a blind VST for exploratory PCA. No `DESeq()` call or statistical test is run.

In [ ]:
make_vsd <- function(counts, metadata) {
  keep <- rowSums(counts >= 10) >= max(2, ceiling(ncol(counts) * 0.10))
  dds <- DESeqDataSetFromMatrix(
    countData = round(counts[keep, , drop = FALSE]),
    colData = metadata,
    design = ~ 1
  )
  list(
    dds = dds,
    vsd = vst(dds, blind = TRUE),
    genes_kept = sum(keep)
  )
}

gill_vsd <- make_vsd(gill_counts, gill_metadata)
liver_vsd <- make_vsd(liver_counts, liver_metadata)
c(gill = gill_vsd$genes_kept, liver = liver_vsd$genes_kept)

In [ ]:
make_pca_data <- function(vsd_obj, metadata) {
  pca <- prcomp(t(assay(vsd_obj)), center = TRUE, scale. = FALSE)
  variance <- 100 * pca$sdev^2 / sum(pca$sdev^2)
  out <- cbind(metadata,
    PC1 = pca$x[, 1], PC2 = pca$x[, 2])
  attr(out, "variance") <- variance
  out
}

gill_pca_data <- make_pca_data(gill_vsd$vsd, gill_metadata)
liver_pca_data <- make_pca_data(liver_vsd$vsd, liver_metadata)

plot_pca <- function(pca_data, title) {
  variance <- attr(pca_data, "variance")
  ggplot(pca_data, aes(PC1, PC2, color = ecotype, shape = true_sex)) +
    geom_point(size = 2.2, na.rm = FALSE) +
    geom_text(
      data = pca_data[pca_data$sample_id == "2024_DJR_196", , drop = FALSE],
      aes(label = sample_id), nudge_y = 0.02, na.rm = TRUE, show.legend = FALSE
    ) +
    facet_wrap(~ population) +
    labs(
      title = title,
      subtitle = "Exploratory VST PCA; no DE model fitted",
      x = paste0("PC1 (", round(variance[1], 1), "%)"),
      y = paste0("PC2 (", round(variance[2], 1), "%)"),
      shape = "WGS/compiled sex"
    ) +
    theme_classic()
}

plot_pca(gill_pca_data, "Gill BRB-seq")
plot_pca(liver_pca_data, "Liver BRB-seq")

### Interpretation note

If `2024_DJR_196` clusters clearly with one sex in the gill PCA, that is useful evidence for follow-up, but it is not by itself a genetic-sex assignment. Population, ecotype, batch, sequencing depth, and sample-quality structure can also drive PCA separation.